# NT Autofix: Greek cleanup + Latvian leftovers + Remove extras

In-place edits on chapter JSONs (like OT `4split_json_upgraded`):

1. **Strip Greek punctuation** — apply `raccs_common` to every `greek` field  
2. **Fix latvian junk** — remove entries that are just punctuation (`,` etc.)  
3. **Add missing leftover_latvian** — compare against lv65 reference  
4. **Remove extra latvian** — words in JSON not found in lv65 reference  
5. **Remove extra greek words** — if JSON has more words than strongs for the verse

## 0 – Config & shared helpers

In [9]:
from pathlib import Path
import json, re, unicodedata, os
from collections import Counter, defaultdict
import pandas as pd

BASE_DIR = Path("bible")
DRY_RUN  = False#True        # True = preview only, False = write files
VERBOSE  = False #True

# ── raccs_common: shared punctuation stripper ──────────────────────────────
def raccs_common(text):
    d = {
        ord('\u2019'): None,  # RIGHT SINGLE QUOTATION MARK  \u2019
        ord('\u2018'): None,  # LEFT SINGLE QUOTATION MARK   \u2018
        ord('\u201C'): None,  # LEFT DOUBLE QUOTATION MARK
        ord('\u201D'): None,  # RIGHT DOUBLE QUOTATION MARK
        ord('['): None,
        ord(']'): None,
        ord('-'): None,
        ord("'"): None,
        ord('\u29FC'): None,  # \u29fc
        ord('\u29FD'): None,  # \u29fd
        ord('*'): None,
        ord('\u21D4'): None,  # \u21d4
        ord('\u3009'): None,  # \u3009
        ord('\u3008'): None,  # \u3008
        ord('\u203F'): None,  # \u203f
        ord('\u00AB'): None,  # \u00ab
        ord('\u00BB'): None,  # \u00bb
        ord('\u2039'): None,  # \u2039
        ord('\u203A'): None,  # \u203a
        ord('('): None,
        ord(')'): None,
        ord(';'): None,
        ord('{'): None,
        ord('}'): None,
    }
    return unicodedata.normalize('NFC', text).translate(d)

# ── Greek / Latvian helpers ───────────────────────────────────────────────
def strip_greek_diacritics(s):
    return ''.join(c for c in unicodedata.normalize('NFD', s)
                   if unicodedata.category(c) != 'Mn')

LV_CHARS = "A-Za-z\u0101\u0113\u012b\u016b\u0123\u0137\u013c\u0146\u0161\u017e\u010d\u0100\u0112\u012a\u016a\u0122\u0136\u013b\u0145\u0160\u017d\u010c"
LV_REGEX = re.compile(f"[{LV_CHARS}]+")

def lv_tokens(text: str):
    return LV_REGEX.findall(unicodedata.normalize('NFC', text))

def strip_lv_diacritics(s: str) -> str:
    return ''.join(c for c in unicodedata.normalize('NFD', s.lower())
                   if unicodedata.category(c) != 'Mn')

## 1 – Load reference datasets

In [10]:
strongs_df = pd.read_csv("strongs.csv")
lv65_df    = pd.read_csv("latvian_full65.csv")

# Pre-clean strongs forms with raccs_common so we compare apples to apples
strongs_df["form_clean"] = strongs_df["form"].astype(str).apply(raccs_common)

strongs_g = strongs_df.groupby(["book", "chapter", "verse"])
lv_g      = lv65_df.groupby(["book", "chapter", "verse"])

print(f"Strongs: {len(strongs_df)} rows, LV65: {len(lv65_df)} rows")

Strongs: 138993 rows, LV65: 7956 rows


## 2 – Dense compact writer (same as OT pattern)

In [11]:
def write_dense_json(file_path, data):
    """
    Write chapter data back in OT-style dense compact format.
    data = list of verse dicts, each with greek_words + leftover_latvian.
    """
    def word_to_compact(w):
        gr  = json.dumps(w.get('greek', ''), ensure_ascii=False)
        lv  = json.dumps(w.get('latvian', []), ensure_ascii=False)
        return f'{{ "greek": {gr}, "latvian": {lv} }}'

    verse_blocks = []
    for verse in data:
        words = verse.get('greek_words', [])
        compact_words = ',\n      '.join(word_to_compact(w) for w in words)
        lo_lv = json.dumps(verse.get('leftover_latvian', []), ensure_ascii=False)
        verse_blocks.append(
            f'  {{\n    "greek_words": [\n      {compact_words}\n    ],\n'
            f'    "leftover_latvian": {lo_lv}\n  }}'
        )

    new_raw = '[\n' + ',\n'.join(verse_blocks) + '\n]'
    with open(file_path, 'w', encoding='utf-8') as f:
        f.write(new_raw)

## 3 – Autofix function

For each verse in a chapter JSON:

**Pass 1 – Greek cleanup:**  
- Apply `raccs_common` to strip `⧼⧽ ‹› 〈〉 {} [] * ‿ '` etc.  
- If word count matches strongs and stripped forms match → accept  
- If JSON has extra greek words not in strongs → remove them  

**Pass 2 – Latvian junk removal:**  
- Remove latvian entries that are pure punctuation (`,` `;` `-` etc.)  

**Pass 3 – Latvian leftovers:**  
- Compare mapped latvian (from greek_words[].latvian + leftover_latvian)  
  against lv65 reference text  
- Add missing ref words to leftover_latvian  
- Remove leftover_latvian entries not in ref (extras/duplicates)

In [12]:
def autofix_chapter(book_name, chapter_num, strongs_g, lv_g, verbose=True, dry_run=True):
    """
    In-place autofix for a single NT chapter JSON.
    Returns (had_fixes: bool, fix_log: list).
    """
    json_path = BASE_DIR / book_name / f"{chapter_num}.json"
    if not json_path.exists():
        return False, []

    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    fixes = []

    for vi, verse_data in enumerate(data):
        verse_num = vi + 1
        key = (book_name, chapter_num, verse_num)
        greek_words = verse_data.get('greek_words', [])
        leftover_latvian = verse_data.get('leftover_latvian', [])

        # ═══════════════════════════════════════════════════════════════════
        # PASS 1: Greek punctuation cleanup via raccs_common
        # ═══════════════════════════════════════════════════════════════════
        for wi, gw in enumerate(greek_words):
            old_gr = gw.get('greek', '')
            new_gr = raccs_common(old_gr).strip()
            if new_gr != old_gr:
                if verbose:
                    print(f"  \u270f\ufe0f\U0001f1ec\U0001f1f7 v{verse_num} w{wi+1}: '{old_gr}' \u2192 '{new_gr}'")
                greek_words[wi]['greek'] = new_gr
                fixes.append(('greek_clean', key, wi+1, old_gr, new_gr))

        # ═══════════════════════════════════════════════════════════════════
        # PASS 1b: Remove extra greek words if JSON has more than strongs
        # ═══════════════════════════════════════════════════════════════════
        if key in strongs_g.groups:
            ref_strongs = strongs_g.get_group(key).sort_values('word')
            ref_count = len(ref_strongs)
            json_count = len(greek_words)

            if json_count > ref_count:
                # Try to identify which extras to remove:
                # build ref stripped forms
                ref_stripped = [strip_greek_diacritics(raccs_common(str(ref_strongs.iloc[i]['form']))).lower()
                                for i in range(ref_count)]
                json_stripped = [strip_greek_diacritics(gw.get('greek', '')).lower()
                                 for gw in greek_words]

                # Find indices in json that don't have a ref match
                # Use a greedy matching: walk ref in order, match to json
                ref_matched = [False] * ref_count
                json_matched = [False] * json_count
                ri = 0
                for ji in range(json_count):
                    if ri < ref_count and json_stripped[ji] == ref_stripped[ri]:
                        json_matched[ji] = True
                        ref_matched[ri] = True
                        ri += 1

                # Only remove if ALL ref words were matched (safe removal)
                if all(ref_matched):
                    extras = [i for i in range(json_count) if not json_matched[i]]
                    if extras:
                        for ei in reversed(extras):  # remove from end
                            removed = greek_words.pop(ei)
                            if verbose:
                                print(f"  \u2796\U0001f1ec\U0001f1f7 v{verse_num} w{ei+1}: removed extra '{removed.get('greek', '')}' lv={removed.get('latvian', [])}")
                            # Salvage any latvian words from the removed entry
                            salvaged_lv = removed.get('latvian', [])
                            if salvaged_lv:
                                leftover_latvian.extend(salvaged_lv)
                        fixes.append(('greek_extra_removed', key, len(extras)))

        # ═══════════════════════════════════════════════════════════════════
        # PASS 2: Latvian junk removal (entries that are just punctuation)
        # ═══════════════════════════════════════════════════════════════════
        for wi, gw in enumerate(greek_words):
            lv_list = gw.get('latvian', [])
            cleaned = [w for w in lv_list if lv_tokens(w)]
            if len(cleaned) != len(lv_list):
                removed_junk = [w for w in lv_list if not lv_tokens(w)]
                if verbose:
                    print(f"  \U0001f5d1\ufe0f\U0001f1f1\U0001f1fb v{verse_num} w{wi+1}: removed junk latvian {removed_junk}")
                greek_words[wi]['latvian'] = cleaned
                fixes.append(('lv_junk', key, wi+1, removed_junk))

        # Also clean leftover_latvian junk
        lo_cleaned = [w for w in leftover_latvian if lv_tokens(w)]
        if len(lo_cleaned) != len(leftover_latvian):
            removed_junk = [w for w in leftover_latvian if not lv_tokens(w)]
            if verbose:
                print(f"  \U0001f5d1\ufe0f\U0001f1f1\U0001f1fb v{verse_num} leftover: removed junk {removed_junk}")
            leftover_latvian = lo_cleaned
            fixes.append(('lv_leftover_junk', key, removed_junk))

        # ═══════════════════════════════════════════════════════════════════
        # PASS 3: Latvian leftovers — add missing, remove extras
        # ═══════════════════════════════════════════════════════════════════
        if key in lv_g.groups:
            lv_text = lv_g.get_group(key).iloc[0]['text']
            ref_lv = lv_tokens(lv_text)

            if ref_lv:
                # Collect all mapped latvian
                all_mapped_lv = []
                for gw in greek_words:
                    for item in gw.get('latvian', []):
                        if item and item != '-':
                            all_mapped_lv.extend(lv_tokens(unicodedata.normalize('NFC', item)))
                for item in leftover_latvian:
                    all_mapped_lv.extend(lv_tokens(unicodedata.normalize('NFC', item)))

                ref_counts    = Counter(ref_lv)
                mapped_counts = Counter(all_mapped_lv)

                # 3a. Find missing ref words → add to leftover_latvian
                new_leftovers = []
                for word, count in ref_counts.items():
                    stripped_w = strip_lv_diacritics(word)
                    matched = sum(v for k, v in mapped_counts.items()
                                  if strip_lv_diacritics(k) == stripped_w)
                    missing_count = count - matched
                    if missing_count > 0:
                        # 'ne-' prefix exception
                        lw = word.lower()
                        if len(lw) > 2 and lw.startswith('ne'):
                            stem_stripped = strip_lv_diacritics(word[2:])
                            stem_matched = sum(v for k, v in mapped_counts.items()
                                               if strip_lv_diacritics(k) == stem_stripped)
                            if stem_matched >= count:
                                continue
                        for _ in range(missing_count):
                            new_leftovers.append(word)

                if new_leftovers:
                    if verbose:
                        print(f"  \u2795\U0001f1f1\U0001f1fb v{verse_num} adding leftovers: {new_leftovers}")
                    leftover_latvian.extend(new_leftovers)
                    fixes.append(('lv_leftover_add', key, new_leftovers))

                # 3b. Remove extra leftover_latvian not in ref
                #     Rebuild: for each leftover word, check if removing it
                #     still keeps us at or above ref counts
                # Re-collect all mapped after additions
                all_mapped_lv2 = []
                for gw in greek_words:
                    for item in gw.get('latvian', []):
                        if item and item != '-':
                            all_mapped_lv2.extend(lv_tokens(unicodedata.normalize('NFC', item)))
                # DON'T include leftover in mapped yet — we'll rebuild it

                mapped_from_greek = Counter(all_mapped_lv2)

                # For each ref word, how many do we still need from leftovers?
                needed_from_leftovers = Counter()
                for word, count in ref_counts.items():
                    stripped_w = strip_lv_diacritics(word)
                    from_greek = sum(v for k, v in mapped_from_greek.items()
                                     if strip_lv_diacritics(k) == stripped_w)
                    need = count - from_greek
                    if need > 0:
                        # ne- prefix exception
                        lw = word.lower()
                        if len(lw) > 2 and lw.startswith('ne'):
                            stem_stripped = strip_lv_diacritics(word[2:])
                            stem_from_greek = sum(v for k, v in mapped_from_greek.items()
                                                  if strip_lv_diacritics(k) == stem_stripped)
                            if stem_from_greek >= count:
                                continue
                        needed_from_leftovers[stripped_w] = need

                # Rebuild leftover_latvian: keep only what's needed
                new_leftover = []
                used_counts = Counter()
                for w in leftover_latvian:
                    sw = strip_lv_diacritics(w)
                    if used_counts[sw] < needed_from_leftovers.get(sw, 0):
                        new_leftover.append(w)
                        used_counts[sw] += 1
                    # else: it's an extra — drop it

                removed_extras = [w for w in leftover_latvian if w not in new_leftover]
                # more precise: track what was dropped
                if len(new_leftover) < len(leftover_latvian):
                    dropped = list(leftover_latvian)  # copy
                    for w in new_leftover:
                        dropped.remove(w)
                    if dropped and verbose:
                        print(f"  \u2796\U0001f1f1\U0001f1fb v{verse_num} removed extra leftovers: {dropped}")
                    leftover_latvian = new_leftover
                    fixes.append(('lv_leftover_trim', key, dropped))

        # Write back into data
        data[vi]['greek_words'] = greek_words
        data[vi]['leftover_latvian'] = leftover_latvian

    # ── Write if any fixes ───────────────────────────────────────────────
    if fixes and not dry_run:
        write_dense_json(json_path, data)
        if verbose:
            print(f"  \u2705 {len(fixes)} fix(es) written to {json_path}")

    return bool(fixes), fixes

## 4 – Discover & run on all chapters

In [13]:
def discover_json_chapters(base_dir):
    """Return dict: book_name -> sorted list of chapter numbers with .json files."""
    result = {}
    for book_dir in sorted(base_dir.iterdir()):
        if not book_dir.is_dir():
            continue
        chapters = sorted(
            int(f.stem) for f in book_dir.iterdir()
            if f.is_file() and f.suffix == '.json' and f.stem.isdigit()
        )
        if chapters:
            result[book_dir.name] = chapters
    return result

book_chapters = discover_json_chapters(BASE_DIR)
total_ch = sum(len(v) for v in book_chapters.values())
print(f"Found {len(book_chapters)} books, {total_ch} chapter JSONs")

Found 27 books, 260 chapter JSONs


In [14]:
# ── Run autofix on all chapters ──────────────────────────────────────────
all_fixes = []
files_changed = 0

for book_name, chapters in book_chapters.items():
    for ch_num in chapters:
        had_fixes, fix_log = autofix_chapter(
            book_name, ch_num, strongs_g, lv_g,
            verbose=VERBOSE, dry_run=DRY_RUN
        )
        if had_fixes:
            files_changed += 1
            all_fixes.extend(fix_log)

action = 'Would fix' if DRY_RUN else 'Fixed'
print(f"\n{'='*60}")
print(f"{action} {files_changed} file(s), {len(all_fixes)} total fix(es).")
if DRY_RUN:
    print("Set DRY_RUN = False and re-run to write changes.")


Fixed 230 file(s), 2603 total fix(es).


## 5 – Fix summary

In [15]:
fix_types = Counter(f[0] for f in all_fixes)
print("Fix breakdown:")
for t, c in fix_types.most_common():
    print(f"  {t}: {c}")

# Per-book summary
book_fix_counts = Counter()
for f in all_fixes:
    if len(f) >= 2 and isinstance(f[1], tuple) and len(f[1]) >= 1:
        book_fix_counts[f[1][0]] += 1

print(f"\nPer-book fix counts:")
for book, count in book_fix_counts.most_common():
    print(f"  {book}: {count}")

Fix breakdown:
  lv_leftover_trim: 1107
  lv_leftover_add: 828
  greek_clean: 628
  lv_leftover_junk: 28
  lv_junk: 10
  greek_extra_removed: 2

Per-book fix counts:
  luke: 437
  matthew: 433
  john: 425
  acts: 389
  mark: 303
  1_corinthians: 207
  2_corinthians: 98
  ephesians: 41
  1_john: 33
  james: 28
  revelation: 24
  colossians: 23
  galatians: 21
  hebrews: 21
  1_thessalonians: 20
  1_peter: 18
  1_timothy: 15
  2_thessalonians: 15
  2_peter: 14
  2_timothy: 12
  romans: 9
  jude: 7
  philippians: 4
  2_john: 3
  3_john: 3


## 6 – (Optional) Run on a single book/chapter for testing

In [16]:
# Test on a single chapter first
# had, log = autofix_chapter('acts', 11, strongs_g, lv_g, verbose=True, dry_run=True)
# print(f"\nFixes: {len(log)}")